# Von Karman vortex street with FEniCSx + `solve_dae`

<a href="https://colab.research.google.com/github/SolveDAE/solve_dae/blob/main/examples/daes/von_karman_vortex_street.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook simulates the classical Karman vortex street: unsteady, incompressible
flow around a cylinder in a channel, following the DFG benchmark case 2D-2
(Reynolds number 100):

> Schaefer, M., Turek, S. (1996). *Benchmark computations of laminar flow around a
> cylinder*. In: Flow Simulation with High-Performance Computers II.
> https://doi.org/10.1007/978-3-322-89849-4_39

The incompressible Navier-Stokes equations are discretized in space with
[FEniCSx](https://fenicsproject.org/) (Taylor-Hood P2/P1 elements) and formulated as
an **index-1 differential algebraic equation (DAE)**: the incompressibility
constraint `div(u) = 0` is enforced algebraically instead of being differentiated,
giving a singular mass matrix `M @ y' = f(t, y)` with `y = (u, p)`. This DAE is
solved in time with [`solve_dae`](https://github.com/SolveDAE/solve_dae)'s implicit
Radau IIA collocation method, using exact Jacobians assembled by FEniCSx.

Running all cells below will:

1. install FEniCSx (via the [FEM on Colab](https://fem-on-colab.github.io/) project),
   `gmsh`, and `solve_dae`,
2. build the mesh, set up and solve the DAE,
3. render an animated GIF of the velocity field and a ParaView-ready VTK time series,
4. zip everything up and offer it for download.

Runtime: a couple of minutes total, most of it spent on the one-time FEniCSx
install.


## 1. Install dependencies

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        import dolfinx
    except ImportError:
        !wget -nc "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh"
        !bash "/tmp/fenicsx-install.sh"

    # the gmsh PyPI wheel is linked against system OpenGL/X11 libraries (for
    # its GUI, which we never use here) that a stock Colab image does not
    # preinstall; without these, `import gmsh` fails with an OSError like
    # "libGLU.so.1: cannot open shared object file".
    !apt-get -qq update && apt-get -qq install -y libglu1-mesa libgl1 libxcursor1 libxinerama1 libxft2 libxrender1

    !pip install -q solve_dae gmsh
else:
    print("Not running on Google Colab -- make sure dolfinx, gmsh, and solve_dae "
          "are already installed in this environment.")


In [ ]:
from importlib.metadata import version, PackageNotFoundError

import dolfinx
import gmsh
import solve_dae


def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "n/a (not pip-installed)"


print(f"dolfinx version:   {dolfinx.__version__}")
print(f"gmsh version:      {pkg_version('gmsh')}")
print(f"solve_dae version: {pkg_version('solve_dae')}")


## 2. Imports

In [ ]:
import time
import numpy as np
from mpi4py import MPI
from petsc4py import PETSc

import basix
import basix.ufl
import dolfinx.fem as dfem
import dolfinx.fem.petsc as dfem_petsc
import dolfinx.io.gmsh as dgmsh
from dolfinx.io import VTKFile
import ufl

from scipy.sparse import csr_matrix
from solve_dae.integrate import solve_dae

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.animation as animation
from IPython.display import HTML, display


## 3. Mesh generation

A channel with a circular obstacle, built with `gmsh`. Facet tags follow the DFG
benchmark convention: 1 = inflow (left), 2 = bottom wall, 3 = outflow (right),
4 = top wall, 5 = cylinder.


In [ ]:
def dolfinx_to_scipy(A):
    """Convert an assembled PETSc matrix to a SciPy CSR matrix."""
    A_csr = A.getValuesCSR()
    return csr_matrix(A_csr[::-1], shape=A.getSize())


def create_mesh(characteristic_length, extends=(2.2, 0.41), center=(0.2, 0.2), radius=0.05):
    """Build a "channel with circular obstacle" mesh using gmsh."""
    gmsh.initialize()
    model = gmsh.model
    model.add("channel_with_cylinder")
    model.setCurrent("channel_with_cylinder")

    dim = 2
    channel = model.occ.addRectangle(0, 0, 0, *extends)
    disk = model.occ.addDisk(*center, 0, radius, radius)
    fluid = model.occ.cut([(dim, channel)], [(dim, disk)])
    model.occ.synchronize()

    volumes = model.getEntities(dim=dim)
    assert volumes == fluid[0]
    fluid_marker = 11
    model.addPhysicalGroup(volumes[0][0], [volumes[0][1]], fluid_marker)
    model.setPhysicalName(volumes[0][0], fluid_marker, "fluid")

    left_marker, bottom_marker, right_marker, top_marker, disk_marker = 1, 2, 3, 4, 5
    boundaries = [
        (left_marker, lambda x: np.isclose(x[0], 0)),
        (bottom_marker, lambda x: np.isclose(x[1], 0)),
        (right_marker, lambda x: np.isclose(x[0], extends[0])),
        (top_marker, lambda x: np.isclose(x[1], extends[1])),
    ]

    left, bottom, right, top = None, None, None, None
    disk_lines = []
    for line in model.getEntities(dim=dim - 1):
        com = model.occ.getCenterOfMass(line[0], line[1])
        for marker, locator in boundaries:
            if locator(com):
                if marker == left_marker:
                    left = line[1]
                elif marker == bottom_marker:
                    bottom = line[1]
                elif marker == right_marker:
                    right = line[1]
                elif marker == top_marker:
                    top = line[1]
                break
        else:
            # none of the straight boundaries matched -> part of the disk
            disk_lines.append(line[1])

    model.addPhysicalGroup(dim - 1, [left], left_marker, "left")
    model.addPhysicalGroup(dim - 1, [bottom], bottom_marker, "bottom")
    model.addPhysicalGroup(dim - 1, [right], right_marker, "right")
    model.addPhysicalGroup(dim - 1, [top], top_marker, "top")
    model.addPhysicalGroup(dim - 1, disk_lines, disk_marker, "disk")

    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", characteristic_length)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", characteristic_length)
    model.mesh.generate(dim=dim)

    mesh_data = dgmsh.model_to_mesh(model, MPI.COMM_WORLD, rank=0, gdim=dim)
    gmsh.finalize()

    markers = dict(left=left_marker, bottom=bottom_marker, right=right_marker,
                   top=top_marker, disk=disk_marker)
    return mesh_data, markers, extends


In [ ]:
# physical parameters (DFG 2D-2 benchmark: mean inflow velocity 1, cylinder
# diameter 0.1, kinematic viscosity 1e-3 -> Reynolds number Re = 1 * 0.1 / 1e-3 = 100)
rho = 1.0
nu = 1.0e-3
Ubar = 1.5  # peak (centerline) inflow velocity

mesh_data, markers, extends = create_mesh(characteristic_length=2.5e-2)
mesh = mesh_data.mesh
facet_tags = mesh_data.facet_tags
gdim = mesh.geometry.dim
print(f"mesh: {mesh.geometry.x.shape[0]} vertices")


## 4. Function spaces and DAE weak form

Taylor-Hood (P2/P1) element pair for velocity/pressure. The mass form `am` has an
empty pressure row (no `dp/dt`), which is what makes this an index-1 DAE rather than
a plain ODE: the incompressibility constraint `div(u) = 0` lives entirely in the
residual `F`.


In [ ]:
# Taylor-Hood element pair (P2 velocity, P1 pressure)
k = 1
Pu = basix.ufl.element("Lagrange", mesh.basix_cell(), k + 1, shape=(gdim,))
Pp = basix.ufl.element("Lagrange", mesh.basix_cell(), k)
V = dfem.functionspace(mesh, basix.ufl.mixed_element([Pu, Pp]))

u, p = ufl.TrialFunctions(V)
v_u, v_p = ufl.TestFunctions(V)

w = dfem.Function(V)   # DAE state y = (u, p)
wp = dfem.Function(V)  # DAE state derivative y' = (u', p')
u_h, p_h = ufl.split(w)

# index-1 DAE: incompressibility div(u) = 0 replaces the (non-existent) time
# derivative of the pressure -> mass matrix `am` has an empty pressure-row
# block, `F` carries the full nonlinear spatial operator.
F = (
    nu * ufl.inner(ufl.grad(v_u), ufl.grad(u_h)) * ufl.dx
    + rho * ufl.inner(v_u, ufl.grad(u_h) * u_h) * ufl.dx
    + ufl.inner(v_p, ufl.div(u_h)) * ufl.dx
)
am = (
    ufl.inner(v_u, u) * ufl.dx
    - ufl.inner(ufl.div(v_u), p) * ufl.dx
)
J = ufl.derivative(F, w)


## 5. Boundary conditions

No-slip on the bottom/top walls and the cylinder, a ramped-up parabolic inflow
profile on the left, and a natural (do-nothing) outflow condition on the right.


In [ ]:
V_u, _ = V.sub(0).collapse()
mesh.topology.create_connectivity(gdim - 1, gdim)

bcs = []

# no-slip: bottom wall, top wall, cylinder
wall_velocity = dfem.Function(V_u)
wall_velocity.x.array[:] = 0.0
for tag in (markers["bottom"], markers["top"], markers["disk"]):
    fcts = facet_tags.find(tag)
    dofs = dfem.locate_dofs_topological((V.sub(0), V_u), gdim - 1, fcts)
    bcs.append(dfem.dirichletbc(wall_velocity, dofs, V.sub(0)))


def smoothstep(x, x_min=0.0, x_max=1.0):
    """C2 smoothstep, used to ramp up the inflow velocity from rest."""
    x = np.clip((x - x_min) / (x_max - x_min), 0.0, 1.0)
    return 6 * x**5 - 15 * x**4 + 10 * x**3


def ramp(t, t_ramp=2.0):
    return smoothstep(t, 0.0, t_ramp)


def inflow_profile(x, t):
    fy = 4.0 * x[1] * (extends[1] - x[1]) / extends[1] ** 2
    return np.stack((Ubar * ramp(t) * fy, np.zeros(x.shape[1])))


inflow_velocity = dfem.Function(V_u)
inflow_velocity.interpolate(lambda x: inflow_profile(x, 0.0))
fcts = facet_tags.find(markers["left"])
dofs = dfem.locate_dofs_topological((V.sub(0), V_u), gdim - 1, fcts)
bcs.append(dfem.dirichletbc(inflow_velocity, dofs, V.sub(0)))
# outflow (right boundary): natural (do-nothing) condition, no BC needed.


## 6. Assembly: residual and Jacobian for `solve_dae`

In [ ]:
form_am = dfem.form(am)
residual_form = dfem.form(F)
jacobian_form = dfem.form(J)

M = dfem_petsc.create_matrix(form_am)
A = dfem_petsc.create_matrix(jacobian_form)

dfem_petsc.assemble_matrix(M, form_am, bcs=bcs)
M.assemble()
M_scipy = dolfinx_to_scipy(M)


def fun_dae(t, y, yp):
    inflow_velocity.interpolate(lambda x: inflow_profile(x, t))

    w.x.array[:] = y
    wp.x.array[:] = yp

    L = dfem_petsc.assemble_vector(residual_form)
    L.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    L.scale(-1)

    A.zeroEntries()
    dfem_petsc.assemble_matrix(A, jacobian_form, bcs=bcs)
    A.assemble()

    # lift the Dirichlet rows: b - J (u_D - u_{k-1}), then set du|_bc = u_D - u_{k-1}
    dfem_petsc.apply_lifting(L, [jacobian_form], [bcs], x0=[w.x.petsc_vec], alpha=1)
    dfem_petsc.set_bc(L, bcs, w.x.petsc_vec, 1.0)
    L.ghostUpdate(addv=PETSc.InsertMode.INSERT_VALUES, mode=PETSc.ScatterMode.FORWARD)

    return M_scipy @ yp - L.array


def jac_dae(t, y, yp):
    w.x.array[:] = y
    wp.x.array[:] = yp

    A.zeroEntries()
    dfem_petsc.assemble_matrix(A, jacobian_form, bcs=bcs)
    A.assemble()

    return dolfinx_to_scipy(A), M_scipy


## 7. Time integration

The fluid starts at rest, which is already a consistent initial state since the
ramped-up inflow is zero at `t = 0`.


In [ ]:
t0, t1 = 0.0, 10.0
w.x.array[:] = 0.0
wp.x.array[:] = 0.0
y0 = w.x.array.copy()
yp0 = wp.x.array.copy()

t_eval = np.linspace(t0, t1, num=300)

start = time.time()
sol = solve_dae(
    fun_dae, (t0, t1), y0, yp0,
    t_eval=t_eval,
    jac=jac_dae,
    method="Radau",
    rtol=1e-4,
    atol=1e-4,
    first_step=1e-3,
)
print(f"elapsed time:  {time.time() - start:.2f} s")
print(f"success:       {sol.success}")
print(f"status:        {sol.status} ({sol.message})")
print(f"nfev / njev / nlu: {sol.nfev} / {sol.njev} / {sol.nlu}")


## 8. VTK output (velocity + pressure), for ParaView

In [ ]:
u_out, p_out = w.sub(0).collapse(), w.sub(1).collapse()
u_out.name, p_out.name = "u", "p"
with (
    VTKFile(mesh.comm, "von_karman_vortex_street_vtk/u.pvd", "w") as vtk_u,
    VTKFile(mesh.comm, "von_karman_vortex_street_vtk/p.pvd", "w") as vtk_p,
):
    for ti, yi in zip(sol.t, sol.y.T):
        w.x.array[:] = yi
        u_out.x.array[:] = w.sub(0).collapse().x.array
        p_out.x.array[:] = w.sub(1).collapse().x.array
        vtk_u.write_function(u_out, ti)
        vtk_p.write_function(p_out, ti)
print("Saved velocity/pressure time series to von_karman_vortex_street_vtk/")


## 9. Animation: velocity magnitude over time

In [ ]:
P1 = dfem.functionspace(mesh, ("Lagrange", 1))
speed_h = dfem.Function(P1)
speed_expr = dfem.Expression(ufl.sqrt(ufl.inner(u_h, u_h)), P1.element.interpolation_points)

coords = P1.tabulate_dof_coordinates()[:, :2]
triangles = mesh.topology.connectivity(gdim, 0).array.reshape(-1, 3)
triangulation = mtri.Triangulation(coords[:, 0], coords[:, 1], triangles)

fig, ax = plt.subplots(figsize=(8, 2))
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
title = ax.set_title("")
levels = np.linspace(0, 2.2 * Ubar, 31)


def update(frame):
    w.x.array[:] = sol.y[:, frame]
    speed_h.interpolate(speed_expr)
    for coll in ax.collections:
        coll.remove()
    ax.tricontourf(triangulation, speed_h.x.array, levels=levels, cmap="viridis", extend="max")
    title.set_text(f"|u|,  t = {sol.t[frame]:.2f} s")


ani = animation.FuncAnimation(fig, update, frames=len(sol.t))
ani.save("von_karman_vortex_street.gif", writer=animation.PillowWriter(fps=20))
plt.close(fig)
print("Saved animation to von_karman_vortex_street.gif")


In [ ]:
# display the animated GIF inline
display(HTML('<img src="von_karman_vortex_street.gif">'))


## 10. Download the results

In [ ]:
import shutil

archive_path = shutil.make_archive("von_karman_vortex_street_results", "zip",
                                    root_dir=".", base_dir="von_karman_vortex_street_vtk")
# also add the GIF to the archive
import zipfile
with zipfile.ZipFile(archive_path, "a") as zf:
    zf.write("von_karman_vortex_street.gif")

print(f"Packaged results into {archive_path}")

if IN_COLAB:
    from google.colab import files
    files.download(archive_path)
else:
    print("Not running on Google Colab -- find the results at:")
    print(f"  {archive_path}")
    print("  von_karman_vortex_street.gif")
    print("  von_karman_vortex_street_vtk/")
